<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/01%20-%20Basics/solutions/01_basics_solutions.ipynb)

# 第一部分 · 练习参考答案

⚠️ **先自己做，再看答案。** 写法不唯一，能得到同样结果的都对。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "01 - Basics/solutions"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


---
## 01a

### 01a-1

In [ ]:
molecules = [
    {"name": "pentacene", "homo": -5.0, "lumo": -3.2},
    {"name": "rubrene",   "homo": -5.4, "lumo": -3.1},
    {"name": "C60",       "homo": -6.1, "lumo": -4.0},
]

rub = molecules[1]
print(f"{rub['name']} 的能隙是 {rub['lumo'] - rub['homo']:.2f} eV")

### 01a-2

In [ ]:
measured = [0.3, 1.2, 0.8, 5.1, 2.2, 0.05, 1.0, 11.0]

count = 0
for mu in measured:
    if mu > 1:
        count += 1
print(count)

# 列表推导式写法
print(len([mu for mu in measured if mu > 1]))
# 更短：True 当作 1 求和
print(sum(mu > 1 for mu in measured))

注意 `1.0` 不算“严格大于 1.0”，所以答案是 4 而不是 5。

### 01a-3

In [ ]:
def classify_gap(gap):
    if gap < 1.0:
        return "narrow"
    elif gap < 3.0:
        return "medium"
    else:
        return "wide"

for mol in molecules:
    gap = mol["lumo"] - mol["homo"]
    print(mol["name"], classify_gap(gap))

print(classify_gap(0.5), classify_gap(1.0), classify_gap(3.4))   # narrow medium wide

---
## 01b

In [ ]:
import os
import pandas as pd

# 本地运行：数据在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮，选择 solubility.csv 即可
DATA_FILE = "../data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请把它放在 data/ 文件夹或和 notebook 同一个文件夹里")

df = pd.read_csv(DATA_FILE)
print("读取成功！一共", len(df), "个分子")

### 01b-1

`df.describe()` 的结果：logS 平均值约 **-2.88**，中位数约 **-2.61**。分子量最大约 **5299**，显然不是小分子（是聚合物/低聚物的一个代表结构）。这提醒我们：**数据集里不一定都是你以为的东西**，用之前要先看。

In [ ]:
df["Solubility"].describe()[["mean", "50%"]], df["MolWt"].max()

### 01b-2

In [ ]:
# 1. 分子量最大的化合物
print(df.loc[df["MolWt"].idxmax(), ["Name", "MolWt", "Solubility"]])

# 2. MolLogP > 5 且 Solubility < -6
print("满足条件个数:", ((df["MolLogP"] > 5) & (df["Solubility"] < -6)).sum())

### 01b-3

In [ ]:
def solubility_class(logs):
    if logs >= 0:
        return "highly soluble"
    elif logs >= -2:
        return "soluble"
    elif logs >= -4:
        return "slightly soluble"
    return "insoluble"

df["sol_class"] = df["Solubility"].apply(solubility_class)
df.groupby("sol_class")[["MolWt", "MolLogP"]].mean().sort_values("MolLogP")

规律：**越难溶的组，平均分子量越大，MolLogP 也越大**（越"亲油"）。

---
## 01c

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


### 01c-1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["TPSA"], bins=50, range=(0, 300))
axes[0].set_xlabel("TPSA (Å²)")
axes[0].set_ylabel("Count")

axes[1].scatter(df["NumRotatableBonds"], df["Solubility"], s=4, alpha=0.3)
axes[1].set_xlim(-1, 40)
axes[1].set_xlabel("Number of rotatable bonds")
axes[1].set_ylabel("logS (log mol/L)")

fig.tight_layout()
plt.show()

### 01c-2

In [ ]:
cols = ["Solubility", "MolWt", "MolLogP", "TPSA", "NumAromaticRings",
        "NumHDonors", "NumHAcceptors", "NumRotatableBonds", "RingCount"]
corr = df[cols].corr()

print(corr["Solubility"].drop("Solubility").abs().sort_values(ascending=False))
print()
print("MolWt vs RingCount:", round(corr.loc["MolWt", "RingCount"], 2))
print("TPSA vs NumHAcceptors:", round(corr.loc["TPSA", "NumHAcceptors"], 2))

1. 与溶解度线性相关性最强的是 **MolLogP**（$r \approx -0.61$），呈**负相关**：越疏水（亲油）越难溶于水。
2. 排序结果见上方代码输出。
3. `MolWt` 与 `RingCount` 的 $r \approx 0.62$，`TPSA` 与 `NumHAcceptors` 的 $r \approx 0.87$。当两个特征高度相关时，它们携带的物理信息有很大一部分是重叠的（称为**多重共线性**）。在线性回归中，这会导致模型的权重系数极不稳定、正负号难以按物理意义直接解释；而在树模型（如随机森林）中，重要度会被两个相关特征平摊。第 2 章会具体演示。

---
## 01d

In [ ]:
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Draw, Descriptors, rdFingerprintGenerator
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

osc_smiles = {
    "thiophene": "c1ccsc1",
    "3-hexylthiophene": "CCCCCCc1ccsc1",
    "2,2'-bithiophene": "c1csc(c1)-c1cccs1",
    "EDOT": "C1COc2cscc2O1",
    "naphthalene": "c1ccc2ccccc2c1",
    "anthracene": "c1ccc2cc3ccccc3cc2c1",
    "tetracene": "c1ccc2cc3cc4ccccc4cc3cc2c1",
    "pentacene": "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
    "TIPS-pentacene": "CC(C)[Si](C#Cc1c2cc3ccccc3cc2c(C#C[Si](C(C)C)(C(C)C)C(C)C)c2cc3ccccc3cc12)(C(C)C)C(C)C",
    "rubrene": "c1ccc(cc1)-c1c2ccccc2c(-c2ccccc2)c2c(-c3ccccc3)c3ccccc3c(-c3ccccc3)c12",
    "BTBT": "c1ccc2c(c1)sc1c2sc2ccccc21",
    "C8-BTBT": "CCCCCCCCc1ccc2c(c1)sc1c3ccc(CCCCCCCC)cc3sc21",
    "perylene": "c1cc2cccc3c4cccc5cccc(c(c1)c23)c54",
    "PTCDA": "O=C1OC(=O)c2ccc3c4ccc5C(=O)OC(=O)c6ccc(c7ccc1c2c37)c4c56",
    "carbazole": "c1ccc2c(c1)[nH]c1ccccc12",
    "triphenylamine": "c1ccc(cc1)N(c1ccccc1)c1ccccc1",
}
osc_mols = {name: Chem.MolFromSmiles(s) for name, s in osc_smiles.items()}

### 01d-1

In [ ]:
acene_names = ["naphthalene", "anthracene", "tetracene", "pentacene"]
mols = [osc_mols[n] for n in acene_names]
legends = [f"{n}\n{CalcMolFormula(m)}" for n, m in zip(acene_names, mols)]
Draw.MolsToGridImage(mols, legends=legends, molsPerRow=4, subImgSize=(250, 200))

### 01d-2

In [ ]:
def canonicalize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)

collected = [
    "c1ccsc1", "C1=CSC=C1", "s1cccc1",
    "c1ccc2ccccc2c1", "C1=CC=C2C=CC=CC2=C1",
    "CCCCCCc1ccsc1", "c1cc(CCCCCC)cs1",
    "c1csc(c1)-c1cccs1",
]
canon = [canonicalize(s) for s in collected]
print(set(canon))
print("不同的分子数:", len(set(canon)))

8 个字符串，只有 **4** 个不同的分子：噻吩、萘、3-己基噻吩、联噻吩。

### 01d-3

In [ ]:
import pandas as pd

fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
names = list(osc_mols.keys())
fps = [fpgen.GetFingerprint(osc_mols[n]) for n in names]
sim = pd.DataFrame([DataStructs.BulkTanimotoSimilarity(fp, fps) for fp in fps], index=names, columns=names)

sim["BTBT"].drop("BTBT").sort_values(ascending=False).head(5)

C8-BTBT 和 BTBT 核心相同，但两条辛基链贡献了很多新的局部环境（CH₃、CH₂、链与环的连接处等），使"至少一方有"的位数大大增加，Tanimoto 的分母变大，所以相似度只有约 0.3。**指纹相似度衡量的是整个分子的片段重合度，不是"核心是否相同"**。想比较核心，先取 Murcko 骨架再算相似度。

In [ ]:
import os
import pandas as pd

# 本地运行：数据在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮，选择 solubility.csv 即可
DATA_FILE = "../data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请把它放在 data/ 文件夹或和 notebook 同一个文件夹里")

df = pd.read_csv(DATA_FILE)
print("读取成功！一共", len(df), "个分子")

In [ ]:
RDLogger.DisableLog("rdApp.*")
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df = df.dropna(subset=["mol"]).reset_index(drop=True)

### 01d-4

In [ ]:
# 1. 分子量是否一致
df["MolWt_rdkit"] = df["mol"].apply(Descriptors.MolWt)
print("最大差值:", (df["MolWt"] - df["MolWt_rdkit"]).abs().max())

# 2. 含 "." 的条目
n_multi = df["SMILES"].apply(lambda s: "." in s).sum()
print("含 '.' 的条目:", n_multi)

1. 最大差值约 0.01，基本一致。微小差别来自不同 RDKit 版本使用的原子量数据。
2. SMILES 里的 `.` 表示**互不相连的几个部分**，通常是盐（阳离子 + 阴离子）、水合物或混合物。数据集里有 1000 多个这样的条目。"一个分子的溶解度"对盐来说其实是**离子化合物的溶解度**，物理机制和中性有机分子不同。很多描述符（比如 logP）对盐也没有意义。实际建模时，常见做法是只保留最大的组分、去掉盐，或者把盐单独处理。

### 01d-5

In [ ]:
anthracene_core = Chem.MolFromSmarts("c1ccc2cc3ccccc3cc2c1")
df["has_anthracene"] = df["mol"].apply(lambda m: m.HasSubstructMatch(anthracene_core))

print("含蒽核:", df["has_anthracene"].sum(), "个")
print(df.groupby("has_anthracene")["Solubility"].mean())

sns.histplot(data=df, x="Solubility", hue="has_anthracene", stat="density", common_norm=False, bins=40, element="step")
plt.xlabel("logS (log mol/L)")
plt.show()

含蒽核的分子平均 logS 低了约 **4 个单位**，即溶解度平均低 **1 万倍**左右。这正是大 π 共轭核心的有机半导体需要接侧链才能溶液加工的原因。